# 01 Collection check and data loading
Loads **all** collected files (old 15-minute days, new 3-minute days, recovered rows) without changing any of them.

In [ ]:
import sys
sys.path.append('analysis')          # run this notebook from the dublinbikes/ folder
import pandas as pd
from load_data import list_status_files, load_all

for f in list_status_files():
    print(f.name)

In [ ]:
df = load_all()          # bike rows + station info + weather (live, or backfill only where blank)
print(df.shape)
print(df['collected_at'].min(), 'to', df['collected_at'].max())
df.head()

In [ ]:
snap = df.drop_duplicates('collected_at').sort_values('collected_at')
snap['gap_min'] = snap['collected_at'].diff().dt.total_seconds() / 60
snap['day'] = snap['collected_at'].dt.strftime('%Y-%m-%d')
snap.groupby('day').agg(snapshots=('collected_at','size'), median_gap=('gap_min','median'),
                        longest_gap=('gap_min','max'))

In [ ]:
# Weather source per snapshot: live / backfill / missing
snap.groupby(['day','weather_source']).size().unstack(fill_value=0)

In [ ]:
# Old 15-minute era vs new 3-minute era (both kept)
snap['era'] = pd.cut(snap['gap_min'], [0, 6, 1e9], labels=['3-minute', 'longer gap'])
snap['era'].value_counts()